# 中文版本

## 数据源介绍
数据来自Kaggle [2015 Flight Delays and Cancellations](https://www.kaggle.com/datasets/usdot/flight-delays)。记录了美国航空公司在2015年的全部航班延误信息。为了更好的和本项目结合，将数据源做下面的修改：
- 将8月1日至12月31日的数据修改为2024年，将1月1日至7月31日的数据修改为2025年，以配合故事中的场景：
  - 故事假设2025年1月获得项目任务，根据2024年4个月的数据预测2025年的航班延误；
  - 每个月都有新的数据，直到项目的重心逐步转到MLOps上，不再关心数据和训练；
  - 1月至7月的数据假设为周期性的真实反馈数据，每周一个文件；

# English Description

## Datasource
The data comes from Kaggle's [2015 Flight Delays and Cancellations](https://www.kaggle.com/datasets/usdot/flight-delays), which records all flight delay information for U.S. airlines in 2015. To better align with the context of this project, the dataset has been modified as follows:
- Data from August 1 to December 31 has been changed to 2024, and data from January 1 to July 31 has been changed to 2025, in order to match the story setting:
  - The story assumes the project task was received in January 2025, and uses four months of data from 2024 to predict flight delays in 2025;
  - New data is available each month, until the project's focus gradually shifts to MLOps, where data and training are no longer the priority;
  - Data from January to July is assumed to be periodic real feedback, with one file per week;


# Start Data Processing

In [2]:
import pandas as pd

import glob
import math
import os


In [ ]:
# download files from S3 bucket, bucket name is in .env

# import os
# from dotenv import load_dotenv

# load_dotenv()
# bucket_name = os.getenv("DWA_BUCKET_NAME")
# s3_file = f"s3://{bucket_name}/storytelling/raw_source/source.tar.gz"
# !aws s3 cp {s3_file} source.tar.gz
# !tar -xzf source.tar.gz -C ../data/source
# !rm source.tar.gz

In [ ]:
df_flights = pd.read_csv('../data/source/flights.csv', low_memory=False)

In [4]:
df_flights.describe()

,YEAR,MONTH,DAY,DAY_OF_WEEK,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,...,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
count,5819079.0,5.819079e+06,5.819079e+06,5.819079e+06,5.819079e+06,5.819079e+06,5.732926e+06,5.732926e+06,5.730032e+06,5.730032e+06,...,5.819079e+06,5.726566e+06,5.714008e+06,5.819079e+06,5.819079e+06,1.063439e+06,1.063439e+06,1.063439e+06,1.063439e+06,1.063439e+06
mean,2015.0,6.524085e+00,1.570459e+01,3.926941e+00,2.173093e+03,1.329602e+03,1.335204e+03,9.370158e+00,1.607166e+01,1.357171e+03,...,1.493808e+03,1.476491e+03,4.407057e+00,2.609863e-03,1.544643e-02,1.348057e+01,7.615387e-02,1.896955e+01,2.347284e+01,2.915290e+00
std,0.0,3.405137e+00,8.783425e+00,1.988845e+00,1.757064e+03,4.837518e+02,4.964233e+02,3.708094e+01,8.895574e+00,4.980094e+02,...,5.071647e+02,5.263197e+02,3.927130e+01,5.102012e-02,1.233201e-01,2.800368e+01,2.143460e+00,4.816164e+01,4.319702e+01,2.043334e+01
min,2015.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-8.200000e+01,1.000000e+00,1.000000e+00,...,1.000000e+00,1.000000e+00,-8.700000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2015.0,4.000000e+00,8.000000e+00,2.000000e+00,7.300000e+02,9.170000e+02,9.210000e+02,-5.000000e+00,1.100000e+01,9.350000e+02,...,1.110000e+03,1.059000e+03,-1.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2015.0,7.000000e+00,1.600000e+01,4.000000e+00,1.690000e+03,1.325000e+03,1.330000e+03,-2.000000e+00,1.400000e+01,1.343000e+03,...,1.520000e+03,1.512000e+03,-5.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,0.000000e+00,2.000000e+00,3.000000e+00,0.000000e+00
75%,2015.0,9.000000e+00,2.300000e+01,6.000000e+00,3.230000e+03,1.730000e+03,1.740000e+03,7.000000e+00,1.900000e+01,1.754000e+03,...,1.918000e+03,1.917000e+03,8.000000e+00,0.000000e+00,0.000000e+00,1.800000e+01,0.000000e+00,1.900000e+01,2.900000e+01,0.000000e+00
max,2015.0,1.200000e+01,3.100000e+01,7.000000e+00,9.855000e+03,2.359000e+03,2.400000e+03,1.988000e+03,2.250000e+02,2.400000e+03,...,2.400000e+03,2.400000e+03,1.971000e+03,1.000000e+00,1.000000e+00,1.134000e+03,5.730000e+02,1.971000e+03,1.331000e+03,1.211000e+03


In [12]:
df_flights.head(5)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2025,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Assume records before August is from 2025 and after is from 2024

In [11]:
# if month >=8 then set year to 2024
df_flights.loc[df_flights['MONTH'] >= 8, 'YEAR'] = 2024
df_flights.loc[df_flights['MONTH'] < 8, 'YEAR'] = 2025

In [13]:
# delete column 'year
df_flights.tail(5)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
5819074,2024,12,31,4,B6,688,N657JB,LAX,BOS,2359,...,753.0,-26.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819075,2024,12,31,4,B6,745,N828JB,JFK,PSE,2359,...,430.0,-16.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819076,2024,12,31,4,B6,1503,N913JB,JFK,SJU,2359,...,432.0,-8.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819077,2024,12,31,4,B6,333,N527JB,MCO,SJU,2359,...,330.0,-10.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819078,2024,12,31,4,B6,839,N534JB,JFK,BQN,2359,...,442.0,2.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Split 2025 data to multiples based on the week number

### 2.1 Delete old data if necessary

In [ ]:
# delete the files YYYY-MM-DOW.csv under ../data/
# import os
# import glob
# files = glob.glob('../data/2025-*.csv')
# for f in files:
#     os.remove(f)

### 2.2 Split data

In [ ]:
# split data of YEAR=2025 based on MONTH and DAY. Every week export to a file, naming pattern: YYYY-MM-Week#.csv. Please calcuate week # based on the year month and day.
df_flights['WEEK'] = pd.to_datetime(df_flights[['YEAR', 'MONTH', 'DAY']]).dt.isocalendar().week

for week in df_flights['WEEK'].unique():
    df_week = df_flights[df_flights['WEEK'] == week]
    df_week.to_csv(f"../data/2025-Week{week:02d}.csv", index=True)


### 2.3 Evaluate the rowcount and variance

In [ ]:
# loop files under ../data/ and print the file name and number of rows in each file

files = glob.glob('../data/2025-*.csv')
file_dict = {}
for f in files:
    df = pd.read_csv(f,low_memory=False)
    file_dict[f] = len(df)

# Find max/min row counts in file_dict
max_file = max(file_dict, key=file_dict.get) # type: ignore
min_file = min(file_dict, key=file_dict.get) # type: ignore
print(f"Max rows file: {max_file} with {file_dict[max_file]} rows")
print(f"Min rows file: {min_file} with {file_dict[min_file]} rows")

# calculate the average and var, evaluate the standard deviation of the row counts
row_counts = list(file_dict.values())
average = sum(row_counts) / len(row_counts)
variance = sum((x - average) ** 2 for x in row_counts) / len(row_counts)
std_dev = math.sqrt(variance)
cv = std_dev / average
print(f"Average rows: {average}")
print(f"Variance: {variance}")
print(f"Standard Deviation: {std_dev}")
print(f"Coefficient of Variation (<0.1 means low variance): {cv}")


Max rows file: ../data/2025-Week31.csv with 135910 rows
Min rows file: ../data/2025-Week5.csv with 104208 rows
Average rows: 111905.36538461539
Variance: 27389994.501109462
Standard Deviation: 5233.545117901389


## 3. Save 2024 data to a file

In [ ]:
# save all 2024 data to one file as the traning source
df_subset_training_source = df_flights[(df_flights['YEAR'] == 2024)]
df_subset_training_source.to_csv("../data/2024.csv", index=True)
print(f"2024 data saved to ../data/2024.csv with {len(df_subset_training_source)} rows")

# 4. Remove non-features from the data
- The inference process will on those files

In [ ]:

fact_files = sorted(glob.glob('../data/2025-Week*.csv'))
i = -1
for f in fact_files:
    i+=1
    df = pd.read_csv(f,low_memory=False)
    # remove column DEPARTURE_TIME
    df = df.drop(columns=['DEPARTURE_TIME','DEPARTURE_DELAY','TAXI_OUT','WHEELS_OFF','ELAPSED_TIME','AIR_TIME','WHEELS_ON','TAXI_IN','ARRIVAL_TIME','ARRIVAL_DELAY','DIVERTED','CANCELLED','CANCELLATION_REASON','AIR_SYSTEM_DELAY','SECURITY_DELAY','AIRLINE_DELAY','LATE_AIRCRAFT_DELAY','WEATHER_DELAY'], errors='ignore',)
    # extract the file name without path and extension
    target_file_name = f.split('/')[-1].split('.')[0]
    # for i=1 to i=5, the files will be in batch_1 folder; for i=6 to i=10, the files will be in batch_2 folder; until for i=51 and i=52, the files will be in batch_11 folder
    batch_number = (i) // 5 + 1
    
    df.to_csv(f'../data/inference/batch_{batch_number}/{target_file_name}.csv', index=True)
    # move the f file to ../data/facts/batch_{batch_number}/
    !mv {f} ../data/facts/batch_{batch_number}/


## 5. [Optional] Upload data to S3 bucket

In [ ]:
def get_bucket_name() -> str:
    bucket_name = os.getenv("DWA_BUCKET_NAME","") 
    if bucket_name == "":
        from dotenv import load_dotenv

        load_dotenv()
        bucket_name = os.getenv("DWA_BUCKET_NAME")

    return bucket_name


In [ ]:
bucket_name = get_bucket_name()

if bucket_name != "":    
    s3_folder = f"s3://{bucket_name}/storytelling/data/"

    folders = ['facts', 'inference', 'training']
    for folder in folders:
        os.system(f"tar -czf {folder}.tar.gz ../data/{folder}/")
        os.system(f"aws s3 cp {folder}.tar.gz {s3_folder}")
        os.system(f"rm {folder}.tar.gz")

upload: ./facts.tar.gz to s3://sagemaker-studio-959750740416-yav0x0ghg3a/storytelling/data/facts.tar.gz
upload: ./inference.tar.gz to s3://sagemaker-studio-959750740416-yav0x0ghg3a/storytelling/data/inference.tar.gz
upload: ./training.tar.gz to s3://sagemaker-studio-959750740416-yav0x0ghg3a/storytelling/data/training.tar.gz
